# Cross-survey **morphology** linear probe — COSMOS ∩ OutThere, each survey separately

Linear-probe the frozen **jwst_dino** teacher on galaxy **morphology**, mirroring astrodino's
`benchmark/linearprobe/linear_probe_morph` and reusing this folder's `linear_probe.py` /
`crossmatch.py`.

**What is different here** — we use *only* the objects imaged by **both** surveys (the OutThere
`sex-*` NIRISS fields overlap COSMOS-Web NIRCam, ~8.7k common objects). Each object has two cutouts
(a COSMOS NIRCam one and an OutThere NIRISS one). We embed each survey **separately**, then fit a
linear probe on each survey's embeddings **on the same object-level train/test split**, and report
COSMOS vs OutThere side by side — i.e. *which survey's image best encodes morphology*.

**Label** — `morph_flag_f150w` (0 spheroid, 1 disk, 2 irregular, 3 bulge+disk; 999999 unclassified)
with confidence `delta_f150w`, from `..._ml_morph.fits` — the exact label this model (jwst_dino) was
probed on before (`CosmosMorphDataset`).

The second half studies **how the data-selection cuts** — confidence `delta`, dropping irregulars,
and a minimum effective-radius (resolved) cut — **change the morph probe** for each survey.

In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from astropy.io import fits
import warnings; warnings.simplefilter('ignore')

sys.path.insert(0, os.path.join('..', '..'))        # jwst_dino/ (model import)
from model.jwst_dino import load_teacher_backbone
from crossmatch import crossmatch_indices, CutoutDataset, extract_embeddings
from dataset import CLASS_NAMES                       # {0:spheroid,1:disk,2:irregular,3:bulge+disk}

ROOT   = '~/ssl_outthere/data/image'
CKPT   = '/home/yacheng/ssl_outthere/encoder_image/jwst_dino/outputs/jwst_dino_ps6_st3/version_6/checkpoints/last.ckpt'
CAT    = '../../../../data/survey/cosmos_2025'
MORPH_CAT = f'{CAT}/COSMOSWeb_mastercatalog_v1_ml_morph.fits'       # morph_flag_f150w, delta_f150w
PHOT_CAT  = f'{CAT}/COSMOSWeb_mastercatalog_v1_photom_primary.fits' # radius_sersic, axratio_sersic (+_err)

TOL          = 0.5            # cross-match radius [arcsec]
PIXSCALE_MAS = 30.0          # cutout pixel scale (both surveys resampled to 30 mas/px)
DEG_TO_PIX   = 3600 * 1000 / PIXSCALE_MAS            # 120000 px per degree (astrodino convention)
SEED         = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## 1) Cross-match the two surveys and pull per-object labels

`crossmatch_indices` returns row-aligned tables (row *i* = the same object in both surveys).
`cos['id']` is the COSMOS master-catalog row index, so it directly indexes all three row-aligned
COSMOS-Web v1 catalogs (no join needed).

In [ ]:
# row-aligned cross-match: cos[i] and out[i] are the SAME object in COSMOS / OutThere
cos, out = crossmatch_indices(ROOT, 'cosmos', 'outthere', tol_arcsec=TOL)
N = len(cos)
print(f'{N} common objects  (median sep {np.median(cos["sep_arcsec"]):.3f}")')
tiles, cnts = np.unique(out['tile'], return_counts=True)
print('OutThere fields:', dict(zip(tiles.tolist(), cnts.tolist())))

ids = np.asarray(cos['id'])                                   # master-catalog row index per object
UNCLASSIFIED = 999999

# morph labels (ml_morph) — what jwst_dino was probed on before
mm = fits.open(os.path.expanduser(MORPH_CAT), memmap=True)[1].data
morph_flag  = np.asarray(mm['morph_flag_f150w'])[ids].astype(int)
morph_delta = np.asarray(mm['delta_f150w'])[ids]

# Sersic effective radius (photom_primary) — used ONLY as a resolved/size selection cut for morph
ph = fits.open(os.path.expanduser(PHOT_CAT), memmap=True)[1].data
re_pix = np.asarray(ph['radius_sersic'], dtype=float)[ids] * DEG_TO_PIX   # r_e in pixels @30mas

print('labels ready:', morph_flag.shape, '| classified:', int((morph_flag != UNCLASSIFIED).sum()))

## 2) Embed each survey separately

Same frozen teacher, same clean preprocessing (`CutoutDataset` = center-crop + asinh). `E_cos[i]`
and `E_out[i]` are the COSMOS and OutThere embeddings of object *i* — perfectly aligned with the
label arrays above.

In [ ]:
net = load_teacher_backbone(CKPT, DEVICE); crop = net.crop_size
E_cos = extract_embeddings(net, CutoutDataset(ROOT, cos, crop_size=crop), DEVICE, num_workers=0)
E_out = extract_embeddings(net, CutoutDataset(ROOT, out, crop_size=crop), DEVICE, num_workers=0)
print('embeddings:', E_cos.shape, E_out.shape)

SURVEYS = {'COSMOS (NIRCam)': E_cos, 'OutThere (NIRISS)': E_out}

## 3) Probe helpers — one shared object split, each survey scored separately

For a given selection we (1) pick the valid objects, (2) make **one** stratified train/test split on
those object indices, then (3) for *each* survey fit a linear probe (`StandardScaler` →
`LogisticRegression(class_weight='balanced')`, the sklearn formulation from `linear_probe.py`) on
that survey's train embeddings and score it on the same test objects. `probe_classification`
**averages over several random splits (seeds)** because the cross-matched morph sample is small and a
single split is noisy. `morph_select` builds the boolean object mask from the data-selection knobs.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def morph_select(delta_max=0.5, exclude_irregular=True, re_min=None):
    """Boolean object mask for the morph probe, built from the data-selection knobs."""
    m = (morph_flag != UNCLASSIFIED) & np.isfinite(morph_delta)
    if delta_max is not None:
        m &= morph_delta < delta_max
    if exclude_irregular:
        m &= morph_flag != 2
    if re_min is not None:
        m &= np.isfinite(re_pix) & (re_pix >= re_min)
    return m

def probe_classification(y, mask, surveys=SURVEYS, test_frac=0.3, seeds=range(5), C=1.0):
    """Fit a linear probe per survey on a shared split; average metrics over `seeds`.

    Returns mean/std accuracy & macro-F1 per survey, plus the last split's predictions
    (for a confusion matrix) and that split's test-label vector.
    """
    y = np.asarray(y); idx = np.where(mask)[0]; yv = y[idx]
    agg = {s: {'acc': [], 'f1': []} for s in surveys}
    last_pred = {}
    for seed in seeds:
        tr, te = train_test_split(np.arange(len(idx)), test_size=test_frac,
                                  random_state=seed, stratify=yv)
        for name, E in surveys.items():
            Xv = E[idx]
            sc = StandardScaler().fit(Xv[tr])
            clf = LogisticRegression(max_iter=3000, C=C, class_weight='balanced')
            clf.fit(sc.transform(Xv[tr]), yv[tr])
            pred = clf.predict(sc.transform(Xv[te]))
            agg[name]['acc'].append(accuracy_score(yv[te], pred))
            agg[name]['f1'].append(f1_score(yv[te], pred, average='macro'))
            last_pred[name] = pred
    res = {'n': len(idx), 'n_test': len(te), 'classes': sorted(np.unique(yv).tolist()),
           'y_test': yv[te], 'last_pred': last_pred}
    for name in surveys:
        res[name] = dict(acc=float(np.mean(agg[name]['acc'])), acc_std=float(np.std(agg[name]['acc'])),
                         f1=float(np.mean(agg[name]['f1'])),   f1_std=float(np.std(agg[name]['f1'])))
    return res

print('probe helpers ready')

## 4) Baseline morph probe  (delta < 0.5, 3-class: spheroid / disk / bulge+disk)

Default selection mirrors the cross-survey embedding notebook: drop unclassified (999999), keep
`delta < 0.5`, drop the irregular class. Imbalance is handled with `class_weight='balanced'`.
Headline metrics are averaged over 5 random splits; the confusion matrices are from the last split.

In [ ]:
mask = morph_select(delta_max=0.5, exclude_irregular=True, re_min=None)
print(f'morph: {mask.sum()} objects')
for k, c in zip(*np.unique(morph_flag[mask], return_counts=True)):
    print(f'  {int(k)} {CLASS_NAMES.get(int(k),"?"):11s}: {c}')

res = probe_classification(morph_flag, mask, seeds=range(5))
classes = res['classes']; names = [CLASS_NAMES.get(c, str(c)) for c in classes]
print(f"\n=== morph probe ({res['n']} objs, {res['n_test']} test, mean±std over 5 splits) ===")
for s in SURVEYS:
    print(f"  {s:18s}  acc={res[s]['acc']:.3f}±{res[s]['acc_std']:.3f}  "
          f"macro-F1={res[s]['f1']:.3f}±{res[s]['f1_std']:.3f}")

fig, ax = plt.subplots(1, len(SURVEYS), figsize=(6*len(SURVEYS), 5))
for a, s in zip(np.atleast_1d(ax), SURVEYS):
    cm = confusion_matrix(res['y_test'], res['last_pred'][s], labels=classes)
    a.imshow(cm, cmap='Blues')
    a.set_xticks(range(len(classes))); a.set_xticklabels(names, rotation=45, ha='right')
    a.set_yticks(range(len(classes))); a.set_yticklabels(names)
    for i in range(len(classes)):
        for j in range(len(classes)):
            a.text(j, i, cm[i, j], ha='center', va='center',
                   color='white' if cm[i, j] > cm.max()/2 else 'black')
    a.set_xlabel('predicted'); a.set_ylabel('true')
    a.set_title(f"{s}\nacc={res[s]['acc']:.3f}, macro-F1={res[s]['f1']:.3f}")
plt.suptitle('Baseline morphology probe — confusion matrices (last split)')
plt.tight_layout(); plt.show()

## 5) Effect of data selection on the morph probe

Vary one selection axis at a time and watch how COSMOS vs OutThere respond:

- **`delta` confidence cut** — stricter = cleaner labels but fewer (and rarer-class-starved) samples.
- **include vs exclude the irregular class** — 4-class vs 3-class problem.
- **minimum effective radius `re_min`** — drop small / unresolved sources, which the shallower,
  lower-resolution OutThere NIRISS images struggle with most.

Each row fits the probe per survey (5-split mean macro-F1 ± std). Configurations whose rarest class
has too few objects to split are skipped.

In [ ]:
# one selection axis varied at a time (baseline = delta<0.5, 3-class)
CONFIGS = [
    ('delta<0.5, 3-class',        dict(delta_max=0.5, exclude_irregular=True)),
    ('delta<0.2, 3-class',        dict(delta_max=0.2, exclude_irregular=True)),
    ('delta<0.1, 3-class',        dict(delta_max=0.1, exclude_irregular=True)),
    ('delta<0.5, 4-class(+irr)',  dict(delta_max=0.5, exclude_irregular=False)),
    ('delta<0.5, 3cls, re>=3px',  dict(delta_max=0.5, exclude_irregular=True, re_min=3)),
    ('delta<0.5, 3cls, re>=5px',  dict(delta_max=0.5, exclude_irregular=True, re_min=5)),
]

snames = list(SURVEYS)
hdr = ' | '.join(f'{s:>18s}' for s in snames)
print(f"{'selection':26s} {'N':>5} {'cls':>4} | {hdr}")
print('-' * (38 + len(hdr)))
sweep = []
for label, kw in CONFIGS:
    m = morph_select(**kw)
    cls, cnt = np.unique(morph_flag[m], return_counts=True)
    if cnt.min() < 12:                       # rarest class too small to stratify-split
        print(f'{label:26s} {int(m.sum()):>5} {len(cls):>4} |  (skipped: rarest class n={cnt.min()})')
        continue
    r = probe_classification(morph_flag, m, seeds=range(5))
    body = ' | '.join(f"F1 {r[s]['f1']:.3f}±{r[s]['f1_std']:.3f}" for s in snames)
    print(f'{label:26s} {int(m.sum()):>5} {len(cls):>4} | {body}')
    sweep.append((label, r))

# bar chart of macro-F1 by selection, COSMOS vs OutThere
labels = [s[0] for s in sweep]; x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
for j, s in enumerate(snames):
    ax.bar(x + (j - 0.5) * w, [r[s]['f1'] for _, r in sweep], w,
           yerr=[r[s]['f1_std'] for _, r in sweep], capsize=3, label=s)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('macro-F1 (mean ± std, 5 splits)')
ax.set_title('Morph probe vs data selection — COSMOS vs OutThere')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6) Summary — COSMOS vs OutThere gap across selections

`gap = COSMOS macro-F1 − OutThere macro-F1` (positive ⇒ COSMOS encodes morphology better).

In [ ]:
print(f"{'selection':26s} {'COSMOS':>8} {'OutThere':>9} {'gap':>7}")
print('-' * 54)
for label, r in sweep:
    c, o = r[snames[0]]['f1'], r[snames[1]]['f1']
    print(f'{label:26s} {c:>8.3f} {o:>9.3f} {c - o:>+7.3f}')

## 7) Across training checkpoints (version_3)

Fix the **baseline** selection (delta < 0.5, 3-class, no `re_min`) and the labels, and swap only the
**encoder checkpoint**. Re-embedding both surveys per checkpoint isolates how SSL training progress
changes how well morphology is linearly decodable — for COSMOS and OutThere separately.

In [ ]:
# Baseline morph probe (delta<0.5, 3-class) across version_3 training checkpoints.
# Each ckpt re-embeds both surveys; the object selection & label are identical, so only
# the encoder changes -> isolates how SSL training progress affects morphology encoding.
import gc

CKPT_DIR = '/home/yacheng/ssl_outthere/encoder_image/jwst_dino/outputs/jwst_dino_ps6_st3/version_5/checkpoints'
CKPTS = {
    'ep10':        f'{CKPT_DIR}/epoch=epoch=9-step=step=5000.ckpt',
    'ep50':        f'{CKPT_DIR}/epoch=epoch=49-step=step=25000.ckpt',
    'ep100':       f'{CKPT_DIR}/epoch=epoch=99-step=step=50000.ckpt',
    'ep150':       f'{CKPT_DIR}/epoch=epoch=149-step=step=75000.ckpt',
    'ep200':       f'{CKPT_DIR}/epoch=epoch=199-step=step=100000.ckpt',
    'ep280(last)': f'{CKPT_DIR}/last.ckpt',
}

ck_mask = morph_select(delta_max=0.5, exclude_irregular=True)        # baseline selection (no re_min)
print(f'baseline morph selection: {ck_mask.sum()} objects\n')

ck_res = {}
for name, path in CKPTS.items():
    net_k = load_teacher_backbone(path, DEVICE); crop_k = net_k.crop_size
    surveys_k = {
        'COSMOS (NIRCam)':   extract_embeddings(net_k, CutoutDataset(ROOT, cos, crop_size=crop_k), DEVICE, num_workers=0),
        'OutThere (NIRISS)': extract_embeddings(net_k, CutoutDataset(ROOT, out, crop_size=crop_k), DEVICE, num_workers=0),
    }
    r = probe_classification(morph_flag, ck_mask, surveys=surveys_k, seeds=range(5))
    ck_res[name] = r
    print(f"{name:12s}  COSMOS F1={r['COSMOS (NIRCam)']['f1']:.3f}±{r['COSMOS (NIRCam)']['f1_std']:.3f}"
          f"   OutThere F1={r['OutThere (NIRISS)']['f1']:.3f}±{r['OutThere (NIRISS)']['f1_std']:.3f}")
    del net_k, surveys_k; gc.collect(); torch.cuda.empty_cache()

# macro-F1 vs checkpoint
labels = list(ck_res); x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(9, 5))
for s in SURVEYS:
    ax.errorbar(x, [ck_res[k][s]['f1'] for k in labels],
                yerr=[ck_res[k][s]['f1_std'] for k in labels],
                marker='o', capsize=3, label=s)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_xlabel('version_3 checkpoint'); ax.set_ylabel('morph macro-F1 (mean ± std, 5 splits)')
ax.set_title('Baseline morph probe vs training checkpoint (version_3)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 7b) version_2 vs version_3

A few version_2 checkpoints under the same baseline selection, overlaid on version_3 to compare the
two training runs.

In [ ]:
# version_2 — a few checkpoints, same baseline selection — overlaid on version_3 for comparison
CKPT_DIR_V2 = '/home/yacheng/ssl_outthere/encoder_image/jwst_dino/outputs/jwst_dino_ps6_st3/version_5/checkpoints'
CKPTS_V2 = {
    'ep10':        f'{CKPT_DIR_V2}/epoch=epoch=9-step=step=6670.ckpt',
    'ep100':       f'{CKPT_DIR_V2}/epoch=epoch=99-step=step=66700.ckpt',
    'ep200':       f'{CKPT_DIR_V2}/epoch=epoch=199-step=step=133400.ckpt',
    'ep219(last)': f'{CKPT_DIR_V2}/last.ckpt',
}

ck_res_v2 = {}
for name, path in CKPTS_V2.items():
    net_k = load_teacher_backbone(path, DEVICE); crop_k = net_k.crop_size
    surveys_k = {
        'COSMOS (NIRCam)':   extract_embeddings(net_k, CutoutDataset(ROOT, cos, crop_size=crop_k), DEVICE, num_workers=0),
        'OutThere (NIRISS)': extract_embeddings(net_k, CutoutDataset(ROOT, out, crop_size=crop_k), DEVICE, num_workers=0),
    }
    r = probe_classification(morph_flag, ck_mask, surveys=surveys_k, seeds=range(5))
    ck_res_v2[name] = r
    print(f"{name:12s}  COSMOS F1={r['COSMOS (NIRCam)']['f1']:.3f}±{r['COSMOS (NIRCam)']['f1_std']:.3f}"
          f"   OutThere F1={r['OutThere (NIRISS)']['f1']:.3f}±{r['OutThere (NIRISS)']['f1_std']:.3f}")
    del net_k, surveys_k; gc.collect(); torch.cuda.empty_cache()

# overlay version_2 vs version_3 vs epoch, one panel per survey
ep_of = lambda lab: int(lab.split('(')[0][2:])      # 'ep280(last)' -> 280
runs = {'version_3': ck_res, 'version_2': ck_res_v2}
fig, axes = plt.subplots(1, len(SURVEYS), figsize=(7*len(SURVEYS), 5), sharey=True)
for a, s in zip(np.atleast_1d(axes), SURVEYS):
    for run, res in runs.items():
        ks = list(res)
        a.errorbar([ep_of(k) for k in ks], [res[k][s]['f1'] for k in ks],
                   yerr=[res[k][s]['f1_std'] for k in ks], marker='o', capsize=3, label=run)
    a.set_title(s); a.set_xlabel('epoch'); a.grid(alpha=0.3); a.legend()
axes[0].set_ylabel('morph macro-F1 (mean ± std, 5 splits)')
plt.suptitle('Baseline morph probe vs checkpoint — version_2 vs version_3')
plt.tight_layout()
plt.show()